In [ ]:
#pip install pandas pyarrow

In [2]:
import pandas as pd

def analyze_parquet(file_path):
    try:
        # 1. Wczytanie pliku
        df = pd.read_parquet(file_path)
        
        print(f"--- Podstawowa analiza pliku: {file_path} ---")
        
        # 2. Wyświetlenie pierwszych kilku wierszy
        print("\n[Podgląd danych - pierwsze 5 wierszy]:")
        print(df.head())
        
        # 3. Informacje o strukturze (typy kolumn, liczba niepustych rekordów)
        print("\n[Informacje o strukturze i typach danych]:")
        print(df.info())
        
        # 4. Podstawowe statystyki opisowe (dla kolumn numerycznych)
        print("\n[Statystyki opisowe]:")
        print(df.describe())
        
        # 5. Sprawdzenie brakujących danych
        print("\n[Liczba brakujących wartości w kolumnach]:")
        print(df.isnull().sum())
        
        # 6. Wymiary ramki danych
        print(f"\nRozmiar zbioru: {df.shape[0]} wierszy i {df.shape[1]} kolumn.")

    except Exception as e:
        print(f"Wystąpił błąd podczas otwierania pliku: {e}")

# Uruchomienie funkcji
analyze_parquet('data_ub.parquet')

--- Podstawowa analiza pliku: data_ub.parquet ---

[Podgląd danych - pierwsze 5 wierszy]:
       OND          MARKET LANGUAGE  \
0  CEG-CAJ  GREAT MERIDIAN  LANG_03   
1  CAJ-BIF  SOUTH MERIDIAN  LANG_09   
2  CEW-BIW    GREAT AURORA  LANG_10   
3  BIC-CAJ      UPPER APEX  LANG_09   
4  CUN-CAJ   NORTH HORIZON  LANG_04   

                                            segments recommendations  \
0  [{"ORIGIN_AIRPORT_CODE":"CEG","DESTINATION_AIR...            [""]   
1  [{"ORIGIN_AIRPORT_CODE":"CAJ","DESTINATION_AIR...            [""]   
2  [{"ORIGIN_AIRPORT_CODE":"CEW","DESTINATION_AIR...            [""]   
3  [{"ORIGIN_AIRPORT_CODE":"BIC","DESTINATION_AIR...            [""]   
4  [{"ORIGIN_AIRPORT_CODE":"CUN","DESTINATION_AIR...            [""]   

                                          emds_order TRIP_START_DATE  \
0  ["MEAL","FAST TRACK","BAGGAGE","SPECIAL EQUIPM...      2038-01-31   
1  ["BUSINESS LOUNGE","FAST TRACK","PET","BAGGAGE...      2038-02-04   
2  ["BAGGAGE","MEAL","FAST

In [3]:
import pandas as pd
import json
import numpy as np

def transform_flight_data(file_path):
    # 1. Wczytanie danych
    print("Wczytywanie danych...")
    df = pd.read_parquet(file_path)
    
    # 2. Konwersja dat
    print("Konwersja dat...")
    df['TRIP_START_DATE'] = pd.to_datetime(df['TRIP_START_DATE'], errors='coerce')
    df['SALES_DATE'] = pd.to_datetime(df['SALES_DATE'], errors='coerce')
    
    # Obliczenie własnego DTD na podstawie dat (weryfikacja ujemnych wartości)
    df['CALCULATED_DTD'] = (df['TRIP_START_DATE'] - df['SALES_DATE']).dt.days
    
    # 3. Obsługa braków danych i anomalii
    print("Uzupełnianie braków danych...")
    # Zastąpienie braków w programie lojalnościowym wartością 'No_Program'
    df['LOYALTY_MEMBERSHIP_PROGRAM'] = df['LOYALTY_MEMBERSHIP_PROGRAM'].fillna('No_Program')
    
    # Przeglądarka - brakujące jako 'Unknown'
    df['BROWSER_TYPE'] = df['BROWSER_TYPE'].fillna('Unknown')
    
    # Zastąpienie brakujących godzin medianą
    median_hour = df['HOUR_OF_THE_DAY_PL_TIME'].median()
    df['HOUR_OF_THE_DAY_PL_TIME'] = df['HOUR_OF_THE_DAY_PL_TIME'].fillna(median_hour)
    
    # Usunięcie lub oflagowanie ujemnych DTD (opcjonalnie można usunąć: df = df[df['DTD'] >= 0])
    df['IS_ANOMALY_DTD'] = df['DTD'] < 0
    
    # 4. Tworzenie nowych cech (Feature Engineering)
    print("Tworzenie nowych cech...")
    df['TOTAL_PASSENGERS'] = df['ADULTS'] + df['TEENAGERS'] + df['CHILDREN'] + df['INFANTS']
    df['HAS_CHILDREN_OR_INFANTS'] = ((df['CHILDREN'] > 0) | (df['INFANTS'] > 0)).astype(int)
    
    # 5. Parsowanie kolumn typu JSON (Wyciąganie wartości z EMD_INFO)
    # EMD (Electronic Miscellaneous Document) to usługi dodatkowe np. bagaż, fast track
    print("Parsowanie struktur JSON...")
    
    def extract_emd_value(emd_string):
        """Zlicza łączną kwotę wydaną na usługi dodatkowe w danej rezerwacji"""
        if pd.isna(emd_string) or emd_string == '' or emd_string == '[""]':
            return 0.0
        try:
            # Niektóre stringi mogą być listami w formacie tekstowym
            records = json.loads(emd_string)
            total_price = 0.0
            for record in records:
                if isinstance(record, dict) and 'emd_price' in record:
                    total_price += float(record['emd_price'])
            return total_price
        except (json.JSONDecodeError, ValueError, TypeError):
            return 0.0

    # Przeliczenie dla pierwszych 10 000 wierszy dla testu (lub usuń limit, aby przeliczyć całość)
    # df['TOTAL_EMD_SPEND'] = df['EMD_INFO'].apply(extract_emd_value) # Użyj tego dla całego zbioru
    
    print("\n--- Zakończono transformacje ---")
    print("\nPrzykładowe nowe kolumny:")
    print(df[['TRIP_START_DATE', 'SALES_DATE', 'CALCULATED_DTD', 'TOTAL_PASSENGERS', 'IS_ANOMALY_DTD']].head())
    
    return df


In [4]:
import pandas as pd
import json
import numpy as np

# Zakładamy, że masz już ramkę danych z poprzedniego kroku o nazwie `df`
df = transform_flight_data('data_ub.parquet')

def advanced_transform_and_analyze(df):
    print("1. Tworzenie zaawansowanych cech czasowych...")
    # Dzień tygodnia i miesiąc wylotu zdradzają, czy to podróż biznesowa, czy wakacyjna
    df['TRIP_MONTH'] = df['TRIP_START_DATE'].dt.month
    df['TRIP_DAY_OF_WEEK'] = df['TRIP_START_DATE'].dt.dayofweek # 0=Poniedziałek, 6=Niedziela
    df['IS_WEEKEND_FLIGHT'] = df['TRIP_DAY_OF_WEEK'].isin([5, 6]).astype(int)
    
    # Sezonowość - przypisanie do kwartału
    df['TRIP_QUARTER'] = df['TRIP_START_DATE'].dt.quarter

    print("2. Analiza głębokich struktur JSON (Złożoność lotu)...")
    def count_segments(segments_str):
        """Zlicza liczbę segmentów lotu (pozwala wykryć przesiadki)"""
        if pd.isna(segments_str) or segments_str == '[""]':
            return 1 # Domyślnie zakładamy lot bezpośredni
        try:
            segments = json.loads(segments_str)
            return len(segments)
        except (json.JSONDecodeError, TypeError):
            return 1

    df['NUMBER_OF_SEGMENTS'] = df['segments'].apply(count_segments)
    df['IS_DIRECT_FLIGHT'] = (df['NUMBER_OF_SEGMENTS'] == 1).astype(int)

    print("3. Wyliczenie całkowitej wartości koszyka (Total Revenue)...")
    # Zakładając, że poprzednio stworzyliśmy kolumnę 'TOTAL_EMD_SPEND'
    # Jeśli nie, wstawiamy 0 jako zabezpieczenie do tego przykładu
    if 'TOTAL_EMD_SPEND' not in df.columns:
        df['TOTAL_EMD_SPEND'] = 0.0 
        
    df['TOTAL_REVENUE'] = df['FARE_YQ_TICKET'] + df['TOTAL_EMD_SPEND']

    print("4. Kategoryzacja okna rezerwacyjnego (Booking Window)...")
    # Zamiast patrzeć na surowe DTD, tworzymy kategorie biznesowe
    bins = [-np.inf, 0, 7, 14, 30, 90, np.inf]
    labels = ['Error/Past', 'Last Minute (0-7)', 'Short (8-14)', 'Medium (15-30)', 'Advance (31-90)', 'Early Bird (90+)']
    df['BOOKING_WINDOW'] = pd.cut(df['CALCULATED_DTD'], bins=bins, labels=labels)

    print("\n--- PODSUMOWANIE ANALITYCZNE ---")
    
    return df

# Uruchomienie (zakładając, że przekazujesz przetworzony wcześniej DataFrame)
df_advanced = advanced_transform_and_analyze(df)

Wczytywanie danych...
Konwersja dat...
Uzupełnianie braków danych...
Tworzenie nowych cech...
Parsowanie struktur JSON...

--- Zakończono transformacje ---

Przykładowe nowe kolumny:
  TRIP_START_DATE SALES_DATE  CALCULATED_DTD  TOTAL_PASSENGERS  IS_ANOMALY_DTD
0      2038-01-31 2037-11-06              86                 3           False
1      2038-02-04 2038-01-25              10                 1           False
2      2038-01-02 2037-12-06              27                 1           False
3      2037-11-16 2037-10-28              19                 1           False
4      2038-02-21 2038-01-27              25                 1           False
1. Tworzenie zaawansowanych cech czasowych...
2. Analiza głębokich struktur JSON (Złożoność lotu)...
3. Wyliczenie całkowitej wartości koszyka (Total Revenue)...
4. Kategoryzacja okna rezerwacyjnego (Booking Window)...

--- PODSUMOWANIE ANALITYCZNE ---


In [8]:
import numpy as np

print("1. Tworzenie behawioralnych profili pasażerów...")

# Czy pasażer leci całkowicie sam? (Solo Traveler)
df_advanced['IS_SOLO_TRAVELER'] = (df_advanced['TOTAL_PASSENGERS'] == 1).astype(int)

# Profil Rodziny (Przynajmniej jeden dorosły + przynajmniej jedno dziecko/niemowlę)
df_advanced['IS_FAMILY'] = ((df_advanced['ADULTS'] > 0) & (df_advanced['HAS_CHILDREN_OR_INFANTS'] == 1)).astype(int)

# Profil Grupy (np. wyjazd integracyjny/znajomi - 3 lub więcej dorosłych, zero dzieci)
df_advanced['IS_ADULT_GROUP'] = ((df_advanced['ADULTS'] >= 3) & (df_advanced['HAS_CHILDREN_OR_INFANTS'] == 0)).astype(int)

# Stosunek dorosłych do reszty pasażerów (pomaga wykryć specyficzne grupy)
df_advanced['ADULT_RATIO'] = np.where(
    df_advanced['TOTAL_PASSENGERS'] > 0, 
    df_advanced['ADULTS'] / df_advanced['TOTAL_PASSENGERS'], 
    0
)

print(df_advanced[['TOTAL_PASSENGERS', 'IS_SOLO_TRAVELER', 'IS_FAMILY', 'IS_ADULT_GROUP']].head())

1. Tworzenie behawioralnych profili pasażerów...
   TOTAL_PASSENGERS  IS_SOLO_TRAVELER  IS_FAMILY  IS_ADULT_GROUP
0                 3                 0          0               1
1                 1                 1          0               0
2                 1                 1          0               0
3                 1                 1          0               0
4                 1                 1          0               0


In [10]:
import json

print("2. Analiza trasy i harmonogramu z segmentów lotu...")

# Rozbicie OND na lotnisko startowe (Origin) i końcowe (Destination)
df_advanced['ORIGIN_AIRPORT'] = df_advanced['OND'].apply(lambda x: x.split('-')[0] if isinstance(x, str) and '-' in x else 'UNKNOWN')
df_advanced['DESTINATION_AIRPORT'] = df_advanced['OND'].apply(lambda x: x.split('-')[1] if isinstance(x, str) and '-' in x else 'UNKNOWN')

# Kategoryzacja pory dnia (lepiej działa dla modeli drzewiastych niż surowa godzina)
bins = [-1, 5, 9, 13, 17, 23]
labels = ['Noc (0-5)', 'Rano (6-9)', 'Południe (10-13)', 'Popołudnie (14-17)', 'Wieczór (18-23)']
df_advanced['FLIGHT_TIME_OF_DAY'] = pd.cut(df_advanced['HOUR_OF_THE_DAY_PL_TIME'], bins=bins, labels=labels)

print(df_advanced[['OND', 'ORIGIN_AIRPORT', 'HOUR_OF_THE_DAY_PL_TIME', 'FLIGHT_TIME_OF_DAY']].head())

2. Analiza trasy i harmonogramu z segmentów lotu...
       OND ORIGIN_AIRPORT  HOUR_OF_THE_DAY_PL_TIME  FLIGHT_TIME_OF_DAY
0  CEG-CAJ            CEG                     14.0  Popołudnie (14-17)
1  CAJ-BIF            CAJ                     12.0    Południe (10-13)
2  CEW-BIW            CEW                     23.0     Wieczór (18-23)
3  BIC-CAJ            BIC                     18.0     Wieczór (18-23)
4  CUN-CAJ            CUN                     19.0     Wieczór (18-23)


In [6]:
df_advanced.to_parquet('transformed_flight_data.parquet')

In [11]:
import pandas as pd
import json

def extract_all_targets(emd_val):
    """
    Przetwarza JSON raz i zwraca słownik ze statusem zakupu dla wszystkich obecnych w nim usług.
    """
    targets = {}
    
    if pd.isna(emd_val) or emd_val == "" or emd_val == '[""]':
        return targets # Zwraca pusty słownik
        
    try:
        # Parsowanie JSON-a
        if isinstance(emd_val, str):
            emd_list = json.loads(emd_val)
        else:
            emd_list = emd_val
            
        # Przejście przez wszystkie usługi w liście
        if isinstance(emd_list, list):
            for item in emd_list:
                if isinstance(item, dict) and 'emd_name' in item:
                    # Zamieniamy nazwę na bezpieczny format kolumny, np. "BUSINESS LOUNGE" -> "TARGET_BUSINESS_LOUNGE"
                    col_name = f"TARGET_{item['emd_name'].replace(' ', '_').upper()}"
                    # Sprawdzamy, czy usługa została sprzedana
                    is_sold = 1 if str(item.get('sold')).lower() == 'true' else 0
                    targets[col_name] = is_sold
    except Exception as e:
        pass
        
    return targets

print("Generowanie kolumn docelowych dla wszystkich usług jednocześnie...")

# 1. Zastosowanie funkcji do kolumny EMD_INFO - to stworzy listę słowników
targets_series = df_advanced['EMD_INFO'].apply(extract_all_targets)

# 2. Zamiana listy słowników na pełnoprawny DataFrame
# Pandas automatycznie dopasuje klucze (nazwy kolumn) do wierszy
targets_df = pd.DataFrame(targets_series.tolist())

# 3. Zastąpienie braków danych (NaN) zerami 
# (Jeśli usługi nie było w JSON-ie wiersza, to na pewno jej nie kupiono)
targets_df = targets_df.fillna(0).astype(int)

# 4. Dołączenie nowych kolumn do głównego DataFrame'u
df_advanced = pd.concat([df_advanced, targets_df], axis=1)

print("\n--- SUKCES ---")
print(f"Wygenerowano następujące kolumny: {', '.join(targets_df.columns)}")

print("\n[Balans klas - odsetek osób kupujących poszczególne usługi]:")
for col in targets_df.columns:
    # Szybki sposób na procentowy udział jedynek w kolumnie 0/1 to obliczenie średniej
    buy_rate = df_advanced[col].mean() * 100
    print(f"{col}: {buy_rate:.2f}%")

Generowanie kolumn docelowych dla wszystkich usług jednocześnie...

--- SUKCES ---
Wygenerowano następujące kolumny: TARGET_BUSINESS_LOUNGE, TARGET_MEAL, TARGET_PET, TARGET_SPECIAL_EQUIPMENT, TARGET_FAST_TRACK, TARGET_BAGGAGE

[Balans klas - odsetek osób kupujących poszczególne usługi]:
TARGET_BUSINESS_LOUNGE: 0.15%
TARGET_MEAL: 0.83%
TARGET_PET: 0.82%
TARGET_SPECIAL_EQUIPMENT: 0.36%
TARGET_FAST_TRACK: 1.52%
TARGET_BAGGAGE: 4.98%


In [12]:
print(df_advanced.columns)

Index(['OND', 'MARKET', 'LANGUAGE', 'segments', 'recommendations',
       'emds_order', 'TRIP_START_DATE', 'SALES_DATE', 'FLIGHT_TYPE', 'DTD',
       'BROWSER_TYPE', 'FIRST_TOUCH_CHANNEL', 'LAST_TOUCH_CHANNEL', 'ADULTS',
       'TEENAGERS', 'CHILDREN', 'INFANTS', 'PAX_TYPE_PASSENGER',
       'HOUR_OF_THE_DAY_PL_TIME', 'LOGIN_STATUS', 'LOYALTY_MEMBERSHIP_PROGRAM',
       'FARE_YQ_TICKET', 'EMD_INFO', 'CALCULATED_DTD', 'IS_ANOMALY_DTD',
       'TOTAL_PASSENGERS', 'HAS_CHILDREN_OR_INFANTS', 'TRIP_MONTH',
       'TRIP_DAY_OF_WEEK', 'IS_WEEKEND_FLIGHT', 'TRIP_QUARTER',
       'NUMBER_OF_SEGMENTS', 'IS_DIRECT_FLIGHT', 'TOTAL_EMD_SPEND',
       'TOTAL_REVENUE', 'BOOKING_WINDOW', 'IS_SOLO_TRAVELER', 'IS_FAMILY',
       'IS_ADULT_GROUP', 'ADULT_RATIO', 'ORIGIN_AIRPORT',
       'DESTINATION_AIRPORT', 'FLIGHT_TIME_OF_DAY', 'TARGET_BUSINESS_LOUNGE',
       'TARGET_MEAL', 'TARGET_PET', 'TARGET_SPECIAL_EQUIPMENT',
       'TARGET_FAST_TRACK', 'TARGET_BAGGAGE'],
      dtype='object')


In [14]:
df_advanced = df_advanced.drop(columns=['EMD_INFO', 'OND', 'segments', 'recommendations',
                                        'emds_order', 'PAX_TYPE_PASSENGER', 'TRIP_START_DATE', 'SALES_DATE'])

In [22]:
import pandas as pd

# Upewniamy się, że usuwamy stary indeks, jeśli się zaplątał
if 'Unnamed: 0' in df_advanced.columns:
    df_advanced = df_advanced.drop(columns=['Unnamed: 0'])

# 2. Łatanie dziur (Imputation)
# Uzupełniamy ewentualne puste wiersze w DTD wartością środkową (medianą)
df_advanced['DTD'] = df_advanced['DTD'].fillna(df_advanced['DTD'].median())

# 3. Zmiana tekstów i kategorii na liczby (One-Hot Encoding)
# Rozszerzamy wyszukiwanie o kolumny typu 'category' (np. BOOKING_WINDOW)
cols_to_encode = df_advanced.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Znaleziono kolumny tekstowe/kategoryczne do zakodowania: {len(cols_to_encode)}")

# Magiczna funkcja get_dummies zamieni teksty i kategorie na kolumny 0/1
df_ml_ready = pd.get_dummies(df_advanced, columns=cols_to_encode, drop_first=True)

# Dla pewności, zamieniamy wszystkie powstałe wartości Prawda/Fałsz (bool) na 1/0
for col in df_ml_ready.select_dtypes(include=['bool']).columns:
    df_ml_ready[col] = df_ml_ready[col].astype(int)

print(f"\n--- GOTOWE! ---")
print(f"Ostateczny rozmiar tabeli: {df_ml_ready.shape}")

# Ostateczne sprawdzenie przed modelem (powinny być zera)
remaining_objects = len(df_ml_ready.select_dtypes(include=['object', 'category']).columns)
print(f"Czy zostały jakieś teksty/kategorie? {remaining_objects} (Powinno być 0!)")
print(f"Czy są jakieś puste wartości? {df_ml_ready.isnull().sum().sum()} (Powinno być 0!)")

# Zapisz swój finalny plik do modelowania (próbka 10 000 wierszy)
df_ml_ready = df_ml_ready.to_parquet('flight_data_ml_ready.parquet', index=False) # Zapisujemy w formacie Parquet bez indeksu

Znaleziono kolumny tekstowe/kategoryczne do zakodowania: 12

--- GOTOWE! ---
Ostateczny rozmiar tabeli: (527947, 474)
Czy zostały jakieś teksty/kategorie? 0 (Powinno być 0!)
Czy są jakieś puste wartości? 0 (Powinno być 0!)


In [18]:
df_advanced_short.to_csv('transformed_flight_data.csv')